# 并发编程与第三方任务工具

这个 notebook 按执行模型组织内容：线程、进程、协程、分布式计算和任务队列。第三方工具章节里的 producer/worker 示例已改成 `src/python_notes` 包布局下的绝对导入，命令也建议从项目根目录执行。


## threading

---

### Event

In [ ]:
from threading import Event, Thread
import time


def countdown(number: int, event: Event) -> None:
    print(f"Countdown from {number} started")
    for index in range(number):
        print(f"Countdown: {number - index}")
        time.sleep(1)
    print(f"Countdown from {number} finished")
    event.set()


event = Event()

thread = Thread(target=countdown, args=(5, event))
thread.start()

event.wait()
print("Countdown completed, main thread continues")

event.clear()

### Condition

In [ ]:
from threading import Condition, Thread
import time as condition_time


def produce(condition: Condition, items: list) -> None:
    for item in range(5):
        with condition:  # 每次生产时短暂持有锁
            items.append(item)
            print(f"Produced {item}")
            condition.notify()
        condition_time.sleep(2)


def consume(condition: Condition, items: list) -> None:
    while True:  # 持续消费
        with condition:
            while not items:
                print("等待生产...")
                condition.wait()
            item = items.pop(0)
            print(f"Consumed {item}")


items = []
condition = Condition()

producer_thread = Thread(target=produce, args=(condition, items))
consumer_thread = Thread(target=consume, args=(condition, items))

producer_thread.start()
consumer_thread.start()

producer_thread.join()


### Lock

In [ ]:
from threading import Lock, Thread
import time as lock_time
from typing import Any


def process(source: Any, lock: Lock) -> None:
    with lock:
        print(f"Processing {source}")
        lock_time.sleep(5)
        print(f"Finished processing {source}")


def get(source: Any, lock: Lock) -> None:
    while True:
        if lock.locked():
            print(f"Wait {source}")
            lock_time.sleep(1)
        else:
            print(f"Get {source}")
            break


lock = Lock()
source = "Data Source"
process_thread = Thread(target=process, args=(source, lock))
get_thread = Thread(target=get, args=(source, lock))
process_thread.start()
get_thread.start()
process_thread.join()
get_thread.join()


### RLock

In [ ]:
from threading import RLock, Thread
import time as rlock_time


def process_data(source: str, lock: RLock) -> None:
    with lock:  # 首次获取
        print(f"🔒 [Process] 开始处理 {source}")
        rlock_time.sleep(1)

        # 嵌套调用
        save_result(source, lock)  # 这里需要再次获取锁
        print(f"✅ [Process] 完成处理 {source}")


def save_result(source: str, lock: RLock) -> None:
    with lock:  # RLock 允许重复获取
        print(f"💾 [Save] 保存 {source} 的结果")
        rlock_time.sleep(0.5)


# 可重入锁
lock = RLock()
source = "关键数据"

t = Thread(target=process_data, args=(source, lock))
t.start()
t.join()


### Lock/RLock

| 特性 | `Lock` (普通锁) | `RLock` (可重入锁) |
| --- | --- | --- |
| **重入性** | ❌ 同一线程无法重复获取锁 | ✅ 同一线程可多次获取锁 |
| **锁的持有者** | 不记录持有线程 | 记录持有线程和递归深度 |
| **释放机制** | 必须由获取锁的线程释放 | 必须由获取锁的线程释放，且释放次数需匹配 |
| **性能** | 更高（简单实现） | 略低（需要维护递归计数） |
| **适用场景** | 简单的互斥操作 | 嵌套/递归调用的代码 |

### 安全使用锁

- 所有线程获取锁的顺序要一致
- 使用超时机制 `acquire(timeout=3)`

### 同时按照顺序获取多个锁的简洁写法

```python
with lock_a, lock_b:
    # 等同于 with lock_a: with lock_b:
    ...
```

### Barrier

In [ ]:
import threading
import time as barrier_time
import random as barrier_random


# 每阶段一个 Barrier
phase1_barrier = threading.Barrier(
    parties=3, action=lambda: print("\n== 所有线程完成数据加载，开始清洗 ==\n")
)
phase2_barrier = threading.Barrier(
    parties=3, action=lambda: print("\n== 所有线程完成数据清洗，开始分析 ==\n")
)


def data_processing(thread_id):
    # 阶段 1：加载
    load_time = barrier_random.uniform(2, 4)
    barrier_time.sleep(load_time)
    print(f"线程 {thread_id} 数据加载完成（耗时 {load_time:.1f}s）")

    phase1_barrier.wait()  # 等待加载完成

    # 阶段 2：清洗
    clean_time = barrier_random.uniform(2, 4)
    barrier_time.sleep(clean_time)
    print(f"线程 {thread_id} 数据清洗完成（耗时 {clean_time:.1f}s）")

    phase2_barrier.wait()  # 等待清洗完成

    # 阶段 3：分析
    analysis_time = barrier_random.uniform(2, 4)
    barrier_time.sleep(analysis_time)
    print(f"线程 {thread_id} 数据分析完成（耗时 {analysis_time:.1f}s）")


# 创建 3 个线程
threads = []
for i in range(1, 4):
    t = threading.Thread(target=data_processing, args=(i,))
    threads.append(t)
    t.start()

for t in threads:
    t.join()

print("所有处理阶段完成！")


### Semaphore

In [ ]:
import threading
import time as semaphore_time
import random as semaphore_random

# 最多 3 个线程同时访问
semaphore = threading.Semaphore(3)


def access_resource(thread_id):
    with semaphore:  # 获取信号量
        print(f"⌛️ Thread-{thread_id} has acquired the resource.")
        semaphore_time.sleep(semaphore_random.uniform(2, 4))  # 模拟耗时
        print(f"✅ Thread-{thread_id} has released the resource.")


threads = []
for i in range(5):
    t = threading.Thread(target=access_resource, args=(i,))
    t.start()
    threads.append(t)

for t in threads:
    t.join()

print("All threads have finished.")


### BoundedSemaphore

In [ ]:
import threading


bounded_semaphore = threading.BoundedSemaphore(3)

try:
    bounded_semaphore.release()
except ValueError as e:
    print(f"Semaphore released failed, {e}")

try:
    bounded_semaphore.acquire(timeout=0.1)
    bounded_semaphore.release()
    print("Semaphore released successfully")
except ValueError as e:
    print(e)

### Semaphore/BoundedSeamphore

| 特性 | `Semaphore` | `BoundedSemaphore` |
| --- | --- | --- |
| **最大信号量数** | 可以随意增加释放次数，不会抛出异常 | 不允许释放次数超过初始化的信号量数 |
| **`release()` 的限制** | 没有限制，可以超过初始计数 | `release()` 次数要与 `acquire()` 一致 |
| **适用场景** | 适合对释放次数没有严格要求的情况 | 适合需要严格控制释放次数的场景，如防止程序中释放信号量过多导致的问题 |

### Timer

In [ ]:
import time as timer_time
import threading
from typing import Any


def do_something(name: Any, queue: list):
    queue.append(name)


pool = list()
timer = threading.Timer(2, do_something, args=("Alice", pool))
timer.start()

while not pool:
    print("Waiting for Alice...")
    timer_time.sleep(1)
else:
    print(pool)


## communication tools

---

### Queue

- `queue.Queue` 普通的 FIFO 队列
- `queue.LifoQueue` LIFO 队列（后进先出）
- `queue.PriorityQueue` 带优先级的队列

In [ ]:
import threading
import queue
import time as queue_time


def produce(production: queue.LifoQueue):
    for goods in range(5):
        queue_time.sleep(1)
        print(f"生产者生产了数据: {goods}")
        production.put(goods)


def consume(production: queue.Queue):
    while True:
        item = production.get()
        if item is None:
            break
        print(f"消费者消费了数据: {item}")
        production.task_done()


production = queue.LifoQueue()
producer_thread = threading.Thread(target=produce, args=(production,))
producer_thread.start()
consumer_thread = threading.Thread(target=consume, args=(production,))
consumer_thread.start()
producer_thread.join()
production.put(None)
consumer_thread.join()
print("所有线程结束")


### threading.local

In [ ]:
import threading
import time as local_time
import random as local_random

# 线程局部存储
local_data = threading.local()


def thread_task(thread_id):
    # 每线程独立
    local_data.value = thread_id
    local_time.sleep(local_random.randrange(1, 3))
    print(f"线程 {thread_id} 的局部数据: {local_data.value}")


threads = []
for i in range(5):
    t = threading.Thread(target=thread_task, args=(i,))
    threads.append(t)
    t.start()

for t in threads:
    t.join()

print("所有线程结束")


## multiprocessing

---

### Pipe

In [ ]:
import multiprocessing


def worker(pipe):
    data = pipe.recv()
    print(f"Received: {data}")
    pipe.send(f"Processed {data}")


def main():
    # 创建管道
    parent, child = multiprocessing.Pipe()

    process = multiprocessing.Process(target=worker, args=(child,))
    process.start()

    # 发送数据
    parent.send("Data from main process")

    # 接收结果
    print(f"Received from worker: {parent.recv()}")

    process.join()


if __name__ == "__main__":
    # spawn 启动
    multiprocessing.set_start_method("spawn")
    main()


### Manager

- `Manager` 下的数据结构是进程间共享的

In [ ]:
import multiprocessing
import random as manager_random
import time as manager_time


def work(share_list, item):
    manager_time.sleep(manager_random.randint(1, 3))
    print(f"I am {item} process, Current list: {share_list}")
    share_list.append(item)


if __name__ == "__main__":
    share_list = multiprocessing.Manager().list()

    processes = []
    for i in range(5):
        p = multiprocessing.Process(target=work, args=(share_list, i))
        processes.append(p)
        p.start()

    for p in processes:
        p.join()

    print(f"Final list: {share_list}")


### Pool

| 方法 | 描述 | 返回值 | 是否异步执行 |
| --- | --- | --- | --- |
| `apply()` | 同步执行单个任务，返回结果 | 任务的结果 | ❌ |
| `apply_async()` | 异步执行单个任务，返回 `AsyncResult` 对象 | `AsyncResult` 对象，稍后获取结果 | ✅ |
| `imap()` | 并行处理可迭代对象，按顺序返回结果 | 迭代器 | ❌ |
| `imap_unordered()` | 并行处理可迭代对象，返回无序结果 | 迭代器 | ❌ |
| `join()` | 等待所有进程完成（通常在调用 `close()` 后使用） | 无 | ❌ |
| `map()` | 并行处理可迭代对象，返回结果（按顺序返回） | 列表 | ❌ |
| `map_async()` | 异步执行 `map()`，返回 `AsyncResult` 对象 | `AsyncResult` 对象，稍后获取结果 | ✅ |
| `Process()` | 直接创建进程并启动，适用于单独的进程任务执行 | 无 | ✅ |
| `starmap()` | 并行处理需要多个参数的任务，解包元组参数 | 列表 | ❌ |
| `starmap_async()` | 异步执行 `starmap()`，返回 `AsyncResult` 对象 | `AsyncResult` 对象，稍后获取结果 | ✅ |
| `terminate()` | 立即终止所有进程，强制结束任务 | 无 | ✅ |
| `close()` | 关闭池，不再接受新的任务 | 无 | ❌ |

In [ ]:
import multiprocessing
import time as pool_time


def square(x):
    pool_time.sleep(1)
    return x * x


if __name__ == "__main__":
    pool = multiprocessing.Pool(processes=4)

    numbers = [1, 2, 3, 4, 5]

    # 无序返回结果
    result = pool.imap_unordered(square, numbers)

    for res in result:
        print(res)

    pool.close()
    pool.join()


### Value/RawValue

| 属性 | `Value` | `RawValue` |
| --- | --- | --- |
| **类型检查** | 提供类型检查，保证数据类型一致 | 不进行类型检查，存储原始字节数据 |
| **内存管理** | 自动加锁，适合常规数据类型（如整数、浮动等） | 无类型管理，更底层，适合存储原始数据 |
| **使用场景** | 适用于共享基本数据类型，如整数、浮动数等 | 适用于需要存储原始数据或更复杂结构的场景 |
| **性能** | 稍慢（有类型检查） | 更快（无类型检查），但需要自己管理数据存储和格式 |


In [ ]:
import multiprocessing
import time as value_time


def increment(shared_value):
    for _ in range(5):
        value_time.sleep(1)
        with shared_value.get_lock():
            shared_value.value += 1
            print(f"Shared value in process: {shared_value.value}")


if __name__ == "__main__":
    # 禁用自动锁，需要自行保证进程安全
    shared_value = multiprocessing.Value("i", 0, lock=False)

    processes = []
    for _ in range(3):
        p = multiprocessing.Process(target=increment, args=(shared_value,))
        p.start()
        processes.append(p)

    for p in processes:
        p.join()

    print(f"Final shared value: {shared_value.value}")


### Array/RawArray

| 属性 | `Array` | `RawArray` |
| --- | --- | --- |
| **类型检查** | 自动提供类型检查，确保数据类型一致 | 不进行类型检查，直接存储原始字节数据 |
| **同步机制** | 自动加锁，保证线程安全 | 不提供自动同步，需要手动加锁 |
| **适用场景** | 适用于需要线程安全访问的共享数组 | 适用于需要低级控制和不需要同步机制的共享数据 |
| **性能** | 稍慢（因为有同步机制） | 更快（因为没有同步机制，适合对数据进行更精细的控制） |

### 自动锁

- 复合语句依旧需要加锁
- 多个变量同步也需要加锁

### 加锁场景

```python
# 非原子操作
with v.get_lock():
    if v.value > 10:
        v.value -= 5

# 多个共享变量
with v1.get_lock(), v2.get_lock():
    v1.value += v2.value
```

In [ ]:
import multiprocessing
import time as array_time


def increment(shared_array, index):
    for _ in range(5):
        array_time.sleep(1)
        shared_array[index] += 1
        print(f"Shared array[{index}] in process: {shared_array[index]}")


if __name__ == "__main__":
    # 启用自动锁
    shared_array = multiprocessing.Array("i", [0, 0, 0], lock=True)

    processes = []
    for i in range(3):
        p = multiprocessing.Process(target=increment, args=(shared_array, i))
        p.start()
        processes.append(p)

    for p in processes:
        p.join()

    print(f"Final shared array: {shared_array[:]}")


### SimpleQueue

In [ ]:
import multiprocessing
import time as simple_queue_time


def producer(q):
    for i in range(5):
        simple_queue_time.sleep(1)
        print(f"Producer putting item {i} into queue")
        q.put(i)  # 向队列中放入数据


def consumer(q):
    for _ in range(5):
        item = q.get()  # 从队列中取出数据
        print(f"Consumer got item {item} from queue")
        simple_queue_time.sleep(2)  # 模拟处理时间


if __name__ == "__main__":
    # 创建一个 SimpleQueue 对象
    q = multiprocessing.SimpleQueue()

    # 创建并启动生产者进程和消费者进程
    producer_process = multiprocessing.Process(target=producer, args=(q,))
    consumer_process = multiprocessing.Process(target=consumer, args=(q,))

    producer_process.start()
    consumer_process.start()

    producer_process.join()  # 等待生产者进程完成
    consumer_process.join()  # 等待消费者进程完成


### Other

- `multiprocessing.Barrier()`
- `multiprocessing.Condition()`
- `multiprocessing.Semaphore()`
- `multiprocessing.BoundedSemaphore()`
- `multiprocessing.Lock()`
- `multiprocessing.RLock()`
- `multiprocessing.Event()`

## concurrent.futures

---

### ThreadPoolExecutor

In [ ]:
import concurrent.futures
import time as thread_pool_time


def fetch_data(x):
    print(f"Fetching data for {x}...")
    thread_pool_time.sleep(2)
    return f"Data {x}"


if __name__ == "__main__":
    with concurrent.futures.ThreadPoolExecutor(max_workers=3) as executor:
        result = list(executor.map(fetch_data, [1, 2, 3, 4, 5]))

    print(result)


### ProcessPoolExecutor

- 等效于 `multiprocessing.Pool()`
- API 更加简洁

In [ ]:
import concurrent.futures
import time as process_pool_time


def compute_square(x):
    print(f"Computing square for {x}...")
    process_pool_time.sleep(2)
    return x * x


if __name__ == "__main__":
    with concurrent.futures.ProcessPoolExecutor(max_workers=3) as executor:
        result = list(executor.map(compute_square, [1, 2, 3, 4, 5]))

    print(result)


## asyncio

---

### async/await

In [ ]:
import asyncio


# 定义一个简单的协程任务
async def hello_world():
    print("Hello")
    await asyncio.sleep(1)  # 模拟一个 I/O 操作
    print("World")


# 运行事件循环
async def main():
    await hello_world()


asyncio.run(main())

In [ ]:
import asyncio


async def fetch_data(x):
    print(f"Fetching data for {x}...")
    await asyncio.sleep(2)
    return f"Data {x}"


async def main():
    tasks = [fetch_data(i) for i in range(5)]
    results = await asyncio.gather(*tasks)
    print(results)


asyncio.run(main())

In [ ]:
import asyncio


async def task(name, delay):
    if delay == 2:
        raise ValueError(f"Task {name} encountered an error!")
    await asyncio.sleep(delay)
    return f"Task {name} completed"


async def main():
    tasks = [task("A", 1), task("B", 2), task("C", 3)]
    # 使用 gather 时捕获异常
    try:
        results = await asyncio.gather(*tasks)
        print(results)
    except Exception as e:
        print(f"Error occurred: {e}")


asyncio.run(main())

In [ ]:
import asyncio


async def periodic_task():
    while True:
        print("Task executed")
        await asyncio.sleep(3)  # 每隔 3 秒执行一次


async def main():
    # 创建周期性任务
    task = asyncio.create_task(periodic_task())
    # 运行 10 秒后停止
    await asyncio.sleep(10)
    task.cancel()  # 取消周期性任务


asyncio.run(main())

In [ ]:
import asyncio

import aiohttp


async def fetch(url):
    async with aiohttp.ClientSession() as session:
        async with session.get(url) as response:
            return await response.text()


async def main():
    urls = ["https://www.example.com", "https://www.python.org"]
    tasks = [fetch(url) for url in urls]
    results = await asyncio.gather(*tasks)
    for result in results:
        print(f"Fetched data: {result}...")


asyncio.run(main())

### 事件循环与结构化并发

现代应用代码通常使用 `asyncio.run()` 启动顶层协程，把事件循环的创建和关闭交给 `asyncio` 管理。只有在框架、REPL 或嵌入式运行时等场景中，才需要直接操作底层 event loop。


In [ ]:
import asyncio


async def timed_task(name: str, delay: float) -> None:
    print(f"Task {name} started")
    await asyncio.sleep(delay)
    print(f"Task {name} completed after {delay} seconds")


async def main() -> None:
    task_a = asyncio.create_task(timed_task("A", 2), name="task-a")
    task_b = asyncio.create_task(timed_task("B", 1), name="task-b")
    await asyncio.gather(task_a, task_b)


asyncio.run(main())


In [ ]:
import asyncio


async def heartbeat() -> None:
    try:
        while True:
            print("Running...")
            await asyncio.sleep(1)
    except asyncio.CancelledError:
        print("Stopped")
        raise


async def main() -> None:
    task = asyncio.create_task(heartbeat())
    await asyncio.sleep(3)
    task.cancel()
    try:
        await task
    except asyncio.CancelledError:
        pass


asyncio.run(main())


### `TaskGroup`

Python 3.11+ 提供 `asyncio.TaskGroup`。它适合“同一作用域内创建的一组任务必须一起完成”的场景：如果其中一个任务失败，任务组会取消其他未完成任务，并把异常按结构化方式抛出。


In [ ]:
import asyncio


async def fetch_user() -> dict[str, str]:
    await asyncio.sleep(0.2)
    return {"id": "U-001", "name": "Ada"}


async def fetch_orders() -> list[str]:
    await asyncio.sleep(0.3)
    return ["ORD-001", "ORD-002"]


async def main() -> None:
    async with asyncio.TaskGroup() as group:
        user_task = group.create_task(fetch_user())
        orders_task = group.create_task(fetch_orders())

    print(user_task.result())
    print(orders_task.result())


asyncio.run(main())


### Queue

In [ ]:
import asyncio


# 生产者协程
async def producer(queue):
    for i in range(5):
        print(f"Producer: producing {i}")
        await queue.put(i)  # 将数据放入队列
        await asyncio.sleep(1)


# 消费者协程
async def consumer(queue):
    while True:
        item = await queue.get()  # 从队列中取出数据
        if item is None:  # 如果是 None，说明生产者已结束，退出消费者
            break
        print(f"Consumer: consumed {item}")
        queue.task_done()  # 标记任务完成
        await asyncio.sleep(2)


async def main():
    # 创建一个容量为 3 的队列
    queue = asyncio.Queue(maxsize=3)

    # 创建生产者和消费者任务
    producer_task = asyncio.create_task(producer(queue))
    consumer_task = asyncio.create_task(consumer(queue))

    # 等待生产者完成
    await producer_task
    # 向消费者发送停止信号（None）
    await queue.put(None)
    # 等待消费者完成
    await consumer_task


# 启动事件循环并执行 main 协程
asyncio.run(main())

### Other

- `asyncio.Barrier()`
- `asyncio.Condition()`
- `asyncio.Semaphore()`
- `asyncio.BoundedSemaphore()`
- `asyncio.Lock()`
- `asyncio.RLock()`
- `asyncio.Event()`

## [Celery](https://docs.celeryq.dev)

Celery 适合稳定的后台任务系统：发送邮件、报表生成、视频转码、异步 Webhook、批量同步等。生产里通常拆成 worker 和 producer 两类进程。

- broker: Redis、RabbitMQ 等消息中间件。
- backend: 保存任务结果，常用 Redis 或数据库。
- worker: 执行任务。
- producer: 提交任务。


### 运行方式

```zsh
celery -A python_notes.examples.task_queues.celery.tasks worker --loglevel=info
python -m python_notes.examples.task_queues.celery.client
```


In [ ]:

import os
from typing import Final

from celery import Celery

DEFAULT_REDIS_URL: Final[str] = "redis://localhost:6379/0"
REDIS_URL: Final[str] = os.getenv("TASK_QUEUE_REDIS_URL", DEFAULT_REDIS_URL)

app = Celery("task_queue_examples", broker=REDIS_URL, backend=REDIS_URL)
app.conf.update(
    task_serializer="json",
    result_serializer="json",
    accept_content=["json"],
    timezone="Asia/Shanghai",
    enable_utc=False,
    task_track_started=True,
    task_time_limit=30,
    result_expires=3600,
)


@app.task(name="notebook.celery.add")
def celery_add(x: int, y: int) -> int:
    """计算两个整数的和。

    Args:
        x: 第一个整数。
        y: 第二个整数。

    Returns:
        两个整数相加后的结果。
    """
    return x + y


@app.task(name="notebook.celery.multiply")
def celery_multiply(x: int, y: int) -> int:
    """计算两个整数的乘积。

    Args:
        x: 第一个整数。
        y: 第二个整数。

    Returns:
        两个整数相乘后的结果。
    """
    return x * y


### Canvas 工作流


In [ ]:

from celery import chain, chord, group

from python_notes.examples.task_queues.celery.tasks import add as celery_add_task, multiply as celery_multiply_task, sum_numbers

single = celery_add_task.apply_async(args=(4, 6), countdown=1)
print("Task ID:", single.id)

chained = chain(celery_add_task.s(2, 3), celery_multiply_task.s(10)).apply_async()
grouped = group(celery_add_task.s(index, index + 1) for index in range(3)).apply_async()
summarized = chord([celery_multiply_task.s(index, 2) for index in range(1, 4)], sum_numbers.s()).apply_async()

print("Chain:", chained.get(timeout=10))
print("Group:", grouped.get(timeout=10))
print("Chord:", summarized.get(timeout=10))


## [Ray](https://www.ray.io)

Ray 更适合 CPU/GPU 计算、模型推理、批处理和 actor 状态管理。它不是传统任务队列，核心能力是把 Python 函数和类变成可分布式调度的任务。


### 集群启动

```zsh
ray start --head --port=6379
ray start --address='192.168.1.100:6379'
```


In [ ]:

import ray

ray.init(ignore_reinit_error=True, include_dashboard=False)


@ray.remote(num_cpus=1)
def score_user(user_id: int) -> dict[str, int]:
    """计算用户分数。

    Args:
        user_id: 用户 ID。

    Returns:
        包含用户 ID 和分数的字典。
    """
    return {"user_id": user_id, "score": user_id * 10}


@ray.remote
class Counter:
    """分布式计数器 actor。"""

    def __init__(self) -> None:
        """初始化计数器。"""
        self.value = 0

    def increment(self, step: int = 1) -> int:
        """递增计数器。

        Args:
            step: 每次增加的步长。

        Returns:
            递增后的计数器值。
        """
        self.value += step
        return self.value


scores = ray.get([score_user.remote(user_id) for user_id in range(5)])
print(scores)

counter = Counter.remote()
print(ray.get([counter.increment.remote() for _ in range(3)]))

ray.shutdown()


## [Dask](https://www.dask.org)

Dask 适合把 Pandas/Numpy 风格的数据计算扩展到多核或多机，也适合构建延迟执行图。它和 Ray 都能做分布式计算，但 Dask 更偏数据分析和任务图。


### 本地集群


In [ ]:

from dask import delayed
from dask.distributed import Client, LocalCluster, as_completed


def load_order(order_id: int) -> dict[str, int]:
    """加载订单数据。

    Args:
        order_id: 订单 ID。

    Returns:
        包含订单 ID 和金额的字典。
    """
    return {"order_id": order_id, "amount": order_id * 100}


def discount(order: dict[str, int]) -> int:
    """计算订单折后金额。

    Args:
        order: 订单数据。

    Returns:
        折后金额。
    """
    return int(order["amount"] * 0.9)


with LocalCluster(n_workers=2, threads_per_worker=1, dashboard_address=None) as cluster:
    with Client(cluster) as client:
        orders = [delayed(load_order)(order_id) for order_id in range(1, 6)]
        amounts = [delayed(discount)(order) for order in orders]
        total = delayed(sum)(amounts)
        print("Delayed Total:", total.compute())

        futures = client.map(load_order, range(6, 9))
        for future in as_completed(futures):
            print("Future Result:", future.result())


### 远程集群

| 角色 | IP 地址 |
| --- | --- |
| Scheduler | 192.168.1.10 |
| Worker 1 | 192.168.1.11 |
| Worker 2 | 192.168.1.12 |

```zsh
# scheduler, display: "Scheduler at: tcp://192.168.1.10:8786"
dask-scheduler

# worker 1 / worker 2
dask-worker tcp://192.168.1.10:8786
```

```python
from dask.distributed import Client

client = Client("tcp://192.168.1.10:8786")
```


## [Dramatiq](https://dramatiq.io)

Dramatiq API 比 Celery 更轻，适合 Redis/RabbitMQ 上的后台任务。常用能力包括 actor、重试、结果存储和 pipeline。

```zsh
dramatiq python_notes.examples.task_queues.dramatiq.tasks
python -m python_notes.examples.task_queues.dramatiq.client
```


In [ ]:

import os
from typing import Final

import dramatiq
from dramatiq.brokers.redis import RedisBroker
from dramatiq.results import Results
from dramatiq.results.backends import RedisBackend

DEFAULT_REDIS_URL: Final[str] = "redis://localhost:6379/0"
REDIS_URL: Final[str] = os.getenv("TASK_QUEUE_REDIS_URL", DEFAULT_REDIS_URL)

result_backend = RedisBackend(url=REDIS_URL)
redis_broker = RedisBroker(url=REDIS_URL)
redis_broker.add_middleware(Results(backend=result_backend))
dramatiq.set_broker(redis_broker)


@dramatiq.actor(actor_name="notebook_dramatiq_add", max_retries=3, min_backoff=1_000, store_results=True)
def dramatiq_add(x: int, y: int) -> int:
    """计算两个整数的和。

    Args:
        x: 第一个整数。
        y: 第二个整数。

    Returns:
        两个整数相加后的结果。
    """
    return x + y


In [ ]:

import dramatiq

from python_notes.examples.task_queues.dramatiq.tasks import add, multiply

message = add.send(3, 5)
print("Message ID:", message.message_id)
print("Result:", message.get_result(block=True, timeout=10_000))

workflow = dramatiq.pipeline([add.message(2, 3), multiply.message(10)]).run()
print("Pipeline Result:", workflow.get_result(block=True, timeout=10_000))


## [ARQ](https://arq-docs.helpmanual.io)

ARQ 基于 asyncio 和 Redis，适合异步 I/O 任务。任务函数是协程，天然适配 HTTP 请求、数据库访问等异步资源。

```zsh
arq python_notes.examples.task_queues.arq.tasks.WorkerSettings
python -m python_notes.examples.task_queues.arq.client
```


In [ ]:

from typing import Any

from arq import cron, func
from arq.connections import RedisSettings
from arq.worker import Retry


async def add(ctx: dict[str, Any], a: int, b: int) -> int:
    """计算两个整数的和。

    Args:
        ctx: ARQ 注入的任务上下文。
        a: 第一个整数。
        b: 第二个整数。

    Returns:
        两个整数相加后的结果。
    """
    return a + b


async def flaky_job(ctx: dict[str, Any], value: int) -> int:
    """演示失败重试。

    Args:
        ctx: ARQ 注入的任务上下文。
        value: 需要处理的整数。

    Returns:
        输入整数的两倍。

    Raises:
        Retry: 前两次执行时触发延迟重试。
    """
    if ctx["job_try"] < 3:
        raise Retry(defer=1)
    return value * 2


async def heartbeat(ctx: dict[str, Any]) -> None:
    """定时输出 worker 心跳。

    Args:
        ctx: ARQ 注入的任务上下文。
    """
    print("ARQ worker heartbeat")


class WorkerSettings:
    """ARQ worker 配置。"""

    functions = [
        func(add, name="add", keep_result=3600, timeout=10),
        func(flaky_job, name="flaky_job", keep_result=3600, timeout=15, max_tries=3),
    ]
    cron_jobs = [cron(heartbeat, minute={0, 30}, run_at_startup=True)]
    redis_settings = RedisSettings(host="localhost", port=6379, database=0)


In [ ]:

import asyncio

from arq import create_pool

from python_notes.examples.task_queues.arq.tasks import redis_config_from_env


async def main() -> None:
    """提交 ARQ 任务并等待结果。"""
    redis = await create_pool(redis_config_from_env().to_settings())
    try:
        job = await redis.enqueue_job("add", 3, 7)
        retried = await redis.enqueue_job("flaky_job", 21)
        print("Job ID:", job.job_id)
        print("Add Result:", await job.result(timeout=5))
        print("Retry Result:", await retried.result(timeout=10))
    finally:
        await redis.close()


asyncio.run(main())


## conclusion

---

### About Queue

- `queue.Queue()` 是线程队列
- `multiprocessing.Queue()` 是进程间队列
- `multiprocessing.SimpleQueue()` 是进程间的简单队列
- `mutliprocessing.Manager().Queue()` 是进程间共享队列
- `asyncio.Queue()` 协程队列

### 进程/线程/协程

| **技术** | **资源开销** | **并行能力** | **适用场景** | **典型用例** |
| --- | --- | --- | --- | --- |
| **进程** | 高 | 是（多核） | CPU 密集型、隔离性要求高 | 视频编码、科学计算 |
| **线程** | 中 | 否（受 GIL） | I/O 密集型、简单数据共享 | 数据库查询、GUI 事件处理 |
| **协程** | 极低 | 否 | 超高并发 I/O、低延迟需求 | Web 服务器、爬虫、聊天服务 |

### 线程/协程（协程为主，线程为辅）

| **维度** | **线程（Thread）** | **协程（Coroutine）** |
| --- | --- | --- |
| **调度机制** | 由操作系统内核调度，抢占式（可能被强制切换） | 用户态协作式调度（需主动 `yield`/`await`） |
| **切换开销** | 高（需内核态/用户态切换，保存寄存器等） | 极低（通常只是函数调用级别的上下文保存） |
| **内存占用** | 每个线程需分配 MB 级栈内存（默认 1-8MB） | 协程栈通常为 KB 级（甚至动态增长） |
| **并行能力** | 受限于 GIL（如 CPython），多核利用率低 | 单线程内并发，无法利用多核 |
| **编程复杂度** | 需处理竞态条件（锁、信号量等） | 无锁编程（但需避免阻塞操作） |

### celery/ray/dask

| 框架 | 适合场景 | 核心特点 |
| --- | --- | --- |
| **Celery** | 异步任务队列（比如发送邮件、处理视频转码） | 分布式任务调度，简单耐用，但主要是 I/O 型任务（不是很强调 CPU 密集计算） |
| **Ray** | 分布式计算（高性能机器学习、大规模 Python 函数调度） | 支持超大规模分布式、远超 Celery 性能，超简单的远程函数（@ray.remote） |
| **Dask** | 分布式数据处理（特别适合 Pandas、Numpy 那种表格/数组运算） | 并行化大数据处理（局部任务调度 + 延迟计算），轻量版的 Spark |

### dramatiq/arq

| 框架 | 适合什么情况 | 特点 |
| --- | --- | --- |
| **dramatiq** | 想要比 Celery 更简单的同步任务队列 | 类似 Celery 但更优雅，支持 RabbitMQ/Redis |
| **arq** | asyncio 项目里需要高性能异步任务队列 | 完全 async/await，超轻量，只用 Redis |